In [5]:
import requests
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NASA_API_KEY")

url = "https://api.nasa.gov/neo/rest/v1/feed"

params = {
    "start_date" : "2026-04-01",
    "end_date" : "2026-04-01",
    "api_key" : API_KEY
}

response = requests.get(url, params = params)

print(response.status_code)
print(type(response.json()))
print(response.json().keys())

200
<class 'dict'>
dict_keys(['links', 'element_count', 'near_earth_objects'])


In [6]:
data = response.json()

# Cuántos asteroides encontró ese día
print("Asteroides ese día:", data['element_count'])

# La data viene organizada por fecha
objetos_por_fecha = data['near_earth_objects']
print("Fechas en el response:", list(objetos_por_fecha.keys()))

# Datos del primer asteriode
primer_asteroide = objetos_por_fecha['2026-04-01'][0]
print("\nCampos disponibles:")
for key, value in primer_asteroide.items():
    print(f"  {key}: {value}")

Asteroides ese día: 16
Fechas en el response: ['2026-04-01']

Campos disponibles:
  links: {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3745999?api_key=p1HfHwH61FZc2aNYvQs7Zb5knPStE0Ds4pHb0JIn'}
  id: 3745999
  neo_reference_id: 3745999
  name: (2016 ES84)
  nasa_jpl_url: https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3745999
  absolute_magnitude_h: 20.31
  estimated_diameter: {'kilometers': {'estimated_diameter_min': 0.2304384666, 'estimated_diameter_max': 0.5152760759}, 'meters': {'estimated_diameter_min': 230.4384665765, 'estimated_diameter_max': 515.2760758959}, 'miles': {'estimated_diameter_min': 0.1431877804, 'estimated_diameter_max': 0.3201776106}, 'feet': {'estimated_diameter_min': 756.031738683, 'estimated_diameter_max': 1690.5383608424}}
  is_potentially_hazardous_asteroid: False
  close_approach_data: [{'close_approach_date': '2026-04-01', 'close_approach_date_full': '2026-Apr-01 17:14', 'epoch_date_close_approach': 1775063640000, 'relative_velocity': {'kilometers_p

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NASA_API_KEY")


def get_asteroids(start_date, end_date):
    """Descarga asteroides entre dos fechas (máximo 7 días por request)"""
    url = "https://api.nasa.gov/neo/rest/v1/feed"
    params = {
        "start_date": start_date,
        "end_date": end_date,
        "api_key": API_KEY
    }
    response = requests.get(url, params=params)
    return response.json()['near_earth_objects']

def extraer_campos(asteroide):
    """Extrae solo los campos que nos interesan en km"""
    close_approach = asteroide['close_approach_data'][0]
    
    return {
        'id': asteroide['id'],
        'name': asteroide['name'],
        'absolute_magnitude_h': asteroide['absolute_magnitude_h'],
        'diameter_min_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_min'],
        'diameter_max_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_max'],
        'velocity_km_per_hour': float(close_approach['relative_velocity']['kilometers_per_hour']),
        'miss_distance_km': float(close_approach['miss_distance']['kilometers']),
        'close_approach_date': close_approach['close_approach_date'],
        'is_potentially_hazardous': asteroide['is_potentially_hazardous_asteroid']
    }

# Descargamos desde enero 2024 hasta 6 Abril 
# La API acepta máximo 7 días por request, así que iteramos de a semanas
start = datetime(2024, 1, 1)
end = datetime(2026, 4, 1)

todos_los_asteroides = []
current = start

print("Descargando datos...")

while current < end:
    siguiente = min(current + timedelta(days=7), end)
    
    datos = get_asteroids(
        current.strftime('%Y-%m-%d'),
        siguiente.strftime('%Y-%m-%d')
    )
    
    for fecha, asteroides in datos.items():
        for asteroide in asteroides:
            todos_los_asteroides.append(extraer_campos(asteroide))
    
    print(f"  ✓ {current.strftime('%Y-%m-%d')} — acumulados: {len(todos_los_asteroides)}")
    
    current = siguiente + timedelta(days=1)
    time.sleep(0.5)  # para no saturar la API

df = pd.DataFrame(todos_los_asteroides)
print(f"\nDataset final: {df.shape[0]} asteroides, {df.shape[1]} columnas")
print(df.head())

Descargando datos...
  ✓ 2024-01-01 — acumulados: 124
  ✓ 2024-01-09 — acumulados: 280
  ✓ 2024-01-17 — acumulados: 420
  ✓ 2024-01-25 — acumulados: 572
  ✓ 2024-02-02 — acumulados: 733
  ✓ 2024-02-10 — acumulados: 888
  ✓ 2024-02-18 — acumulados: 1022
  ✓ 2024-02-26 — acumulados: 1188
  ✓ 2024-03-05 — acumulados: 1317
  ✓ 2024-03-13 — acumulados: 1444
  ✓ 2024-03-21 — acumulados: 1578
  ✓ 2024-03-29 — acumulados: 1732
  ✓ 2024-04-06 — acumulados: 1888
  ✓ 2024-04-14 — acumulados: 2069
  ✓ 2024-04-22 — acumulados: 2217
  ✓ 2024-04-30 — acumulados: 2359
  ✓ 2024-05-08 — acumulados: 2505
  ✓ 2024-05-16 — acumulados: 2617
  ✓ 2024-05-24 — acumulados: 2735
  ✓ 2024-06-01 — acumulados: 2864
  ✓ 2024-06-09 — acumulados: 2965
  ✓ 2024-06-17 — acumulados: 3083
  ✓ 2024-06-25 — acumulados: 3204
  ✓ 2024-07-03 — acumulados: 3318
  ✓ 2024-07-11 — acumulados: 3420
  ✓ 2024-07-19 — acumulados: 3527
  ✓ 2024-07-27 — acumulados: 3643
  ✓ 2024-08-04 — acumulados: 3772
  ✓ 2024-08-12 — acumulados: 3903